# 경제 분석 및 예측과 데이터 지능 실습2 (final)

이 노트북은 다음 두 실습을 하나로 통합합니다.

- (A) 일별 방문자 수 예측 (trend/seasonality/CV)
- (B) Kaggle Store Sales 데이터 EDA + 예측 (exogenous 포함)

가능한 한 sktime로 구성하지만, Prophet/NeuralProphet은 설치되어 있을 때만 실행됩니다.

In [ ]:
# --- Import hygiene (repo contains ./sktime source checkout) ---
# Running from repo root can shadow `import sktime` via ./sktime (namespace).
# Force using the actual package at ./sktime/sktime by adjusting sys.path.
import os
import sys

repo_root = os.path.abspath(os.getcwd())
local_sktime_src = os.path.join(repo_root, "sktime")
if os.path.exists(os.path.join(local_sktime_src, "sktime", "__init__.py")):
    sys.path = [p for p in sys.path if p not in ("", repo_root)]
    sys.path.insert(0, local_sktime_src)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.model_selection import (
    temporal_train_test_split,
    SlidingWindowSplitter,
)
from sktime.forecasting.model_evaluation import evaluate
from sktime.performance_metrics.forecasting import (
    mean_absolute_error,
    mean_absolute_percentage_error,
)
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.compose import ForecastingPipeline, RecursiveTabularRegressionForecaster
from sktime.transformations.series.fourier import FourierFeatures
from sktime.transformations.series.boxcox import LogTransformer

plt.rcParams["figure.figsize"] = (11, 4)

def score(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": float(mean_absolute_percentage_error(y_true, y_pred)),
    }

## A) Daily website visitors

In [ ]:
path = "../datasets"
raw = pd.read_csv(os.path.join(path, "daily-website-visitors.csv"), thousands=",")
raw["Date"] = pd.to_datetime(raw["Date"])
raw = raw[["Date", "First.Time.Visits"]].rename(columns={"First.Time.Visits": "y"})
raw["y"] = pd.to_numeric(raw["y"], errors="coerce")
raw = raw.dropna().sort_values("Date")
raw = raw[raw["Date"] >= pd.to_datetime("2017-01-01")].copy()

y = raw.set_index("Date")["y"].asfreq("D")
y = y.interpolate(limit_direction="both")
y.head()

In [ ]:
y.plot(title="Daily website visitors")
plt.show()

In [ ]:
y_train, y_test = temporal_train_test_split(y, test_size=int(len(y) * 0.15))
fh = ForecastingHorizon(y_test.index, is_relative=False)
len(y_train), len(y_test)

### Baselines

In [ ]:
baselines = {
    "Naive_last": NaiveForecaster(strategy="last"),
    "Naive_mean": NaiveForecaster(strategy="mean"),
    "Naive_seasonal_7": NaiveForecaster(strategy="last", sp=7),
}

preds = {}
for name, f in baselines.items():
    f.fit(y_train)
    preds[name] = f.predict(fh)

ax = y_train.iloc[-120:].plot(label="train", alpha=0.7)
y_test.plot(ax=ax, label="test", alpha=0.9)
for name, yhat in preds.items():
    yhat.plot(ax=ax, label=name)
ax.legend()
ax.set_title("Baselines")
plt.show()

pd.DataFrame({k: score(y_test, v) for k, v in preds.items()}).T.sort_values("MAPE")

### Feature-based model: Fourier seasonality + recursive regression

In [ ]:
model_ts = ForecastingPipeline(
    steps=[
        ("log", LogTransformer()),
        (
            "fourier",
            FourierFeatures(
                sp_list=[7, 365.25],
                fourier_terms_list=[3, 5],
                keep_original_columns=False,
            ),
        ),
        (
            "reg",
            RecursiveTabularRegressionForecaster(
                estimator=RandomForestRegressor(
                    n_estimators=300,
                    random_state=42,
                    n_jobs=-1,
                ),
                window_length=28,
            ),
        ),
    ]
)

model_ts.fit(y_train)
y_pred = model_ts.predict(fh)

ax = y_train.iloc[-180:].plot(label="train", alpha=0.7)
y_test.plot(ax=ax, label="test", alpha=0.9)
y_pred.plot(ax=ax, label="Fourier+RF(recursive)")
ax.legend()
ax.set_title("Fourier + recursive regression")
plt.show()

score(y_test, y_pred)

### Time series cross-validation

In [ ]:
cv = SlidingWindowSplitter(
    fh=np.arange(1, 31),
    window_length=365,
    step_length=60,
)

cv_results = evaluate(
    model_ts,
    y=y,
    cv=cv,
    strategy="refit",
    scoring=[mean_absolute_error, mean_absolute_percentage_error],
    return_data=False,
)
metric_cols = [c for c in cv_results.columns if c.startswith("test_")]
cv_results[metric_cols].describe()

### (Optional) Prophet / NeuralProphet

In [ ]:
prophet_available = False
neuralprophet_available = False
try:
    import prophet  # noqa: F401
    prophet_available = True
except Exception:
    prophet_available = False

try:
    import neuralprophet  # noqa: F401
    neuralprophet_available = True
except Exception:
    neuralprophet_available = False

prophet_available, neuralprophet_available

In [ ]:
if prophet_available:
    from sktime.forecasting.fbprophet import Prophet as SktimeProphet

    p = SktimeProphet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        n_changepoints=10,
    )
    p.fit(y_train)
    yhat_p = p.predict(fh)
    print(score(y_test, yhat_p))
else:
    print("Prophet not installed; skipping.")

## B) Kaggle Store Sales

In [ ]:
base = "../datasets/kaggle_store_sales"
paths = {
    "train": os.path.join(base, "train.csv"),
    "test": os.path.join(base, "test.csv"),
    "stores": os.path.join(base, "stores.csv"),
    "transactions": os.path.join(base, "transactions.csv"),
    "oil": os.path.join(base, "oil.csv"),
    "holidays": os.path.join(base, "holidays_events.csv"),
}

has_kaggle = all(os.path.exists(p) for p in paths.values())
has_kaggle

In [ ]:
if has_kaggle:
    train = pd.read_csv(paths["train"], parse_dates=["date"])
    oil = pd.read_csv(paths["oil"], parse_dates=["date"])
    display(train.head())
else:
    idx = pd.date_range("2017-01-01", periods=365 * 3, freq="D")
    rng = np.random.default_rng(42)
    y_k = (
        200
        + 0.02 * np.arange(len(idx))
        + 10 * np.sin(2 * np.pi * np.arange(len(idx)) / 7)
        + 30 * np.sin(2 * np.pi * np.arange(len(idx)) / 365.25)
        + rng.normal(0, 5, size=len(idx))
    )
    y_k = pd.Series(y_k, index=idx, name="sales")
    X_k = pd.DataFrame(
        {
            "oil": 50 + 3 * np.sin(2 * np.pi * np.arange(len(idx)) / 365.25) + rng.normal(0, 0.8, size=len(idx)),
            "onpromotion": (rng.random(len(idx)) < 0.1).astype(int),
        },
        index=idx,
    )
    print("Kaggle data not found. Using synthetic series.")
    display(y_k.head())

In [ ]:
if has_kaggle:
    total_by_day = train.groupby("date")["sales"].sum().sort_index()
    total_by_day.plot(title="Total sales (all stores/families)")
    plt.show()
else:
    y_k.plot(title="Synthetic sales")
    plt.show()

### Single-series forecasting with exogenous X

In [ ]:
if has_kaggle:
    one = train[(train["store_nbr"] == 1) & (train["family"] == "GROCERY I")].copy().sort_values("date")
    y_k = one.set_index("date")["sales"].asfreq("D").astype(float).interpolate(limit_direction="both")

    oil_ = oil.set_index("date")["dcoilwtico"].asfreq("D").interpolate(limit_direction="both")
    promo_ = one.set_index("date")["onpromotion"].asfreq("D").fillna(0)
    X_k = pd.DataFrame({"oil": oil_, "onpromotion": promo_}, index=y_k.index).interpolate(limit_direction="both")

y_train_k, y_test_k, X_train_k, X_test_k = temporal_train_test_split(y_k, X_k, test_size=90)
fh_k = ForecastingHorizon(y_test_k.index, is_relative=False)

baseline_k = NaiveForecaster(strategy="last", sp=7)
baseline_k.fit(y_train_k)
yhat_base_k = baseline_k.predict(fh_k)

model_k = ForecastingPipeline(
    steps=[
        ("fourier", FourierFeatures(sp_list=[7, 365.25], fourier_terms_list=[3, 5])),
        (
            "reg",
            RecursiveTabularRegressionForecaster(
                estimator=RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
                window_length=28,
            ),
        ),
    ]
)

model_k.fit(y_train_k, X=X_train_k)
yhat_k = model_k.predict(fh_k, X=X_test_k)

ax = y_train_k.iloc[-180:].plot(label="train", alpha=0.7)
y_test_k.plot(ax=ax, label="test", alpha=0.9)
yhat_base_k.plot(ax=ax, label="SeasonalNaive(sp=7)")
yhat_k.plot(ax=ax, label="Fourier+RF(recursive)")
ax.legend()
ax.set_title("Store sales: single-series with exogenous X")
plt.show()

print("MAPE baseline:", float(mean_absolute_percentage_error(y_test_k, yhat_base_k)))
print("MAPE model:", float(mean_absolute_percentage_error(y_test_k, yhat_k)))

## Appendix: Fourier features

In [ ]:
ff = FourierFeatures(sp_list=[7, 365.25], fourier_terms_list=[1, 1])
X_fourier = ff.fit_transform(y_train_k)
X_fourier.head()